# Entregable Transformers
### Text generation with Transformer
##### Trabajo realizado por: Manuel Enciso Martínez

**Descripción:** Vamos a reutilizar la práctica de clase sobre Implementar un Transformer **causal** como capa de Keras y usarlo para generación de texto sobre letras de rap (https://www.kaggle.com/datasets/smunoz3801/9325-letras-de-rap-en-espaol) y recetas de cocina (https://huggingface.co/datasets/somosnlp/RecetasDeLaAbuela), ambos repositorios en castellano.

## Instalaciones

In [ ]:
!pip install tensorflow
!pip install keras --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.6 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.13.2
    Uninstalling keras-3.13.2:
      Successfully uninstalled keras-3.13.2


In [ ]:
import keras
from keras import ops
from keras import layers
import numpy as np
import pandas as pd
import requests, io
from collections import Counter

## Implementacion del bloque Transformer como capa

In [ ]:
class TransformerBlock(layers.Layer):
    """
    Implementa un bloque Transformer CAUSAL como una capa de Keras.
    Hereda de layers.Layer, podemos usarlo igual que cualquier
    otra capa de Keras: modular y reutilizable.

    La mascara causal impide que cada token vea tokens futuros,
    lo que es esencial para la generacion autoregresiva de texto.

    Parametros:
    -----------
    embed_dim : int
        Dimension del espacio de embedding.

    num_heads : int
        Numero de "cabezas" de atencion en la Multi-Head Attention.

    ff_dim : int
        Tamano de la capa oculta en la red Feed-Forward.

    rate : float
        Tasa de Dropout.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        # Llamamos al constructor de la clase padre (layers.Layer)
        # Esto inicializa todas las estructuras internas de Keras
        super().__init__()

        # --- SUB-CAPA 1: Multi-Head Self-Attention ----------------------------
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,   # Numero de cabezas de atencion en paralelo
            key_dim=embed_dim      # Dimension de los vectores Query/Key
        )

        # --- SUB-CAPA 2: Feed-Forward Network (FFN) ---------------------------
        # Red completamente conectada que se aplica a CADA POSICION por separado.
        # La primera capa expande la dimension (captura patrones complejos)
        # La segunda la contrae de vuelta a embed_dim (para mantener consistencia)
        self.ffn = keras.Sequential([  #capa fullconected despues del multihead conected
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        # --- CAPAS DE NORMALIZACION -------------------------------------------
        # Normaliza las activaciones para estabilizar el entrenamiento.
        self.layernorm1 = layers.LayerNormalization()  # Despues de attention
        self.layernorm2 = layers.LayerNormalization()  # Despues de FFN

        # --- DROPOUT ----------------------------------------------------------
        self.dropout1 = layers.Dropout(rate)  # Dropout tras la attention
        self.dropout2 = layers.Dropout(rate)  # Dropout tras la FFN
        #* Se usan layernorm_i, dropout_i porque se aplican en vectores diferentes y el gradiente al actualizar necesita capas diferentes

    def call(self, inputs):
          # Input tensor de forma (batch_size, seg_legm embed_dim): cuantos bloques de txt, longitud maxima de bloque texto, cuantas dimensiones para representacion de tokens

        """
        Define el flujo de datos a traves del bloque (forward pass).

        use_causal_mask=True: aplica una mascara triangular inferior para que
        cada token solo pueda atender a posiciones ANTERIORES (no al futuro).
        Es la unica diferencia respecto al Transformer de clasificacion.

        inputs: tensor de forma (batch_size, seq_len, embed_dim)
        """
        attn_output = self.att(inputs, inputs, use_causal_mask=True)  # Self-attention causal
        attn_output = self.dropout1(attn_output)

        out1 = self.layernorm1(inputs + attn_output) # Lo que se recibe de la capa de atencion mas la entrada original (conexion residual)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)

        return self.layernorm2(out1 + ffn_output) # Suma de antes de ffn (feed forward network) y el resultado de depues



## Implementación capa de embedding

Dos capas de embedding separadas, una para token y otra para indices (posición).

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """
    Capa de embedding que combina:
    1. Embedding de tokens
    2. Embedding de posicion

    Parametros:
    -----------
    maxlen : int
        Longitud maxima de la secuencia de entrada.

    vocab_size : int
        Tamano del vocabulario (numero de palabras unicas que el modelo conoce).

    embed_dim : int
        Dimension del vector de embedding.
    """
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()

        # --- EMBEDDING DE TOKENS ----------------------------------------------
        # Esta tabla se inicializa aleatoriamente y se APRENDE durante el entrenamiento.
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)

        # --- EMBEDDING DE POSICION --------------------------------------------
        # Esta tabla se inicializa aleatoriamente y se APRENDE durante el entrenamiento.
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x): # Cuidado con x no es vector, cuando salga si
        """
        Forward pass de la capa de embedding.

        x: tensor de indices de tokens, forma (batch_size, seq_len)
           Cada numero es el indice de una palabra en el vocabulario.
           Ej: [1, 14, 3, 22, 0, 0, ...] donde 0 = padding

        Devuelve: tensor de forma (batch_size, seq_len, embed_dim)
        """
        # ops.shape(x)[-1]: cogemos la ultima dimension de la forma de x,
        # que es la longitud de la secuencia.
        # Usamos ops.shape (y no x.shape) para que funcione en modo dinamico.
        maxlen = ops.shape(x)[-1]

        # ops.arange genera [0, 1, 2, ..., maxlen-1]
        # Estos indices enteros se pasan al embedding de posicion
        positions = ops.arange(start=0, stop=maxlen, step=1)

        # Convertimos cada posicion en su vector de embedding
        # Forma de salida: (maxlen, embed_dim)
        positions = self.pos_emb(positions)

        # Para cada indice en x, busca el vector correspondiente en la tabla
        # Forma de salida: (batch_size, seq_len, embed_dim)
        x = self.token_emb(x)

        # El broadcasting se encarga de la suma:
        # x tiene forma (batch_size, maxlen, embed_dim)
        # positions tiene forma (maxlen, embed_dim)
        # -> Se suma la misma posicion para todas las muestras del batch
        return x + positions

## Descargar y preparar dataset

### Canciones de rap

Lo tenemos descargado en local y subimos el archivo al google colab

In [ ]:
df = pd.read_csv('corpus_rap.csv', on_bad_lines='skip')  # Archivo subido a colab
print(df.shape)
print(df.head())

(9325, 7)
   id           artista                                            cancion  \
0   0             Denom             Machete (con Jarfaiter y Gente jodida)   
1   1             Denom                           Vacío (con Ivo Incuerdo)   
2   2             Denom  El orgullo es fiel (con Juancho Marqués y Elio...   
3   3             Denom                    Mueve mueve (con Fernandocosta)   
4   4  Jaro Desperdizio                                           Insomnia   

                           album  \
0                       Medicina   
1                       Medicina   
2                       Medicina   
3                       Medicina   
4  Sin álbum, es un vídeo suelto   

                                               letra  anyo  visitas  
0  Para su nuevo disco "Medicina", Denom ha vuelt...  2019      126  
1  [Denom]\nYo que quería, yo que pedía vida,\nSe...  2019      361  
2  "El orgullo es fiel" es uno de los cortes incl...  2019      262  
3  [Estribillo: Denom] (

Observamos que son más de $9000$ canciones de rap y la letra (nuestro objetivo para generar canciones) está en la columna 'letra'

In [ ]:
vocab_size = 30000  # Solo consideramos las 20k->30k palabras mas frecuentes
maxlen = 64        # Ventana de contexto: 100->64 palabras
step = 8            # Desplazamiento entre secuencias (cuanto mayor, menos solapamiento)

# --- TOKENIZACION A NIVEL DE PALABRA -----------------------------------------
# Concatenamos todos los versiculos en un unico texto y dividimos por espacios
df["letra"] = df["letra"].astype(str)
full_text = " ".join(df["letra"].fillna("").astype(str)).lower()
all_words  = full_text.split()

Mostremos las primeras 100 palabras

In [ ]:
all_words[:100]

['para',
 'su',
 'nuevo',
 'disco',
 '"medicina",',
 'denom',
 'ha',
 'vuelto',
 'a',
 'contar',
 'con',
 'la',
 'colaboración',
 'vocal',
 'de',
 'jarfaiter,',
 'con',
 'quién',
 'ya',
 'ha',
 'colaborado',
 'anteriormente',
 'en',
 'el',
 'tema',
 '"machete";',
 'corte',
 'en',
 'el',
 'que',
 'tambien',
 'participa',
 'el',
 'grupo',
 'gente',
 'jodida.',
 'de',
 'la',
 'instrumental',
 'se',
 'encarga',
 'narcosbeats.',
 '¿tienes',
 'ya',
 'la',
 'letra',
 'para',
 'este',
 'tema?',
 'ayúdanos',
 'y',
 '¡envíanosla!',
 '[denom]',
 'yo',
 'que',
 'quería,',
 'yo',
 'que',
 'pedía',
 'vida,',
 'se',
 'partió',
 'por',
 'la',
 'mitad',
 'lo',
 'que',
 'tenía',
 'y',
 'ahora',
 'es',
 'frio,',
 'te',
 'siento',
 'dentro',
 'mío,',
 'como',
 'el',
 'dolor,',
 'como',
 'una',
 'tenía,',
 'tragos',
 'de',
 'colonia,',
 'principios',
 'de',
 'histeria,',
 'no',
 'hay',
 'color,',
 'yo',
 'me',
 'elija',
 'a',
 'no',
 'llorar,',
 'aunque',
 'no',
 'sonría,']

In [ ]:
# Vocabulario: indice 0 = <PAD>, indice 1 = <UNK>, luego las mas frecuentes
word_counts = Counter(all_words)
vocab   = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(vocab_size - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("Tamaño vocabulario total:\n",len(word_counts))


Tamaño vocabulario total:
 228224


In [ ]:

# Convertimos a indices (palabras fuera del vocabulario -> 1 = <UNK>)
token_ids = [word2idx.get(w, 1) for w in all_words]

# --- CONSTRUCCION DE SECUENCIAS (X, Y) ---------------------------------------
# Cada muestra de entrenamiento:
#   x = token_ids[i : i+maxlen]       <- contexto de entrada
#   y = token_ids[i+1 : i+maxlen+1]   <- siguiente token en cada posicion
# El modelo aprende: dado el token t, predice el token t+1.
sequences = [token_ids[i:i+maxlen+1]
             for i in range(0, len(token_ids) - maxlen - 1, step)]
sequences = np.array(sequences)

x_all = sequences[:, :-1]   # (N, maxlen)
y_all = sequences[:, 1:]    # (N, maxlen) -- siguiente token en cada posicion

# Particion 80/20 train/val
split = int(0.8 * len(x_all))
x_train, y_train = x_all[:split], y_all[:split]
x_val,   y_val   = x_all[split:], y_all[split:]

print(len(x_train), "Training sequences")
print(len(x_val),   "Validation sequences")

443534 Training sequences
110884 Validation sequences


In [ ]:
# Decodificamos la primera secuencia de entrada
decoded_sequence = " ".join(idx2word.get(i, '<UNK>') for i in x_train[0])
decoded_sequence

'para su nuevo disco <UNK> <UNK> ha vuelto a contar con la colaboración vocal de <UNK> con quién ya ha <UNK> <UNK> en el tema <UNK> corte en el que tambien <UNK> el grupo gente <UNK> de la instrumental se encarga <UNK> ¿tienes ya la letra para este tema? ayúdanos y ¡envíanosla! [denom] yo que quería, yo que pedía vida, se partió por la'

In [ ]:
# La etiqueta correspondiente: cada token es el siguiente al de x_train[0]
" ".join(idx2word.get(i, '<UNK>') for i in y_train[0])

'su nuevo disco <UNK> <UNK> ha vuelto a contar con la colaboración vocal de <UNK> con quién ya ha <UNK> <UNK> en el tema <UNK> corte en el que tambien <UNK> el grupo gente <UNK> de la instrumental se encarga <UNK> ¿tienes ya la letra para este tema? ayúdanos y ¡envíanosla! [denom] yo que quería, yo que pedía vida, se partió por la mitad'

### Creación modelo de lenguaje

El Transformer procesa la secuencia entera con mascara causal y en **cada posicion** predice el siguiente token.

La capa de salida es `Dense(vocab_size, softmax)` aplicada a los `maxlen` pasos temporales. Esto hace que aumente considerablemente los parámetros a entrenar en función del tamaño del vocabulario.

In [ ]:
embed_dim = 32  # Embedding size for each token
num_heads = 2   # Number of attention heads
ff_dim = 32     # Hidden layer size in feed forward network inside transformer

inputs = layers.Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)

# Salida en CADA posicion temporal -> (batch, maxlen, vocab_size)
# No hay pooling: predecimos el siguiente token para cada token de la secuencia.

outputs = layers.Dense(vocab_size, activation="softmax")(x)
#

model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 64, 32)         │       962,048 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 64, 32)         │        10,656 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64, 30000)      │       990,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,962,704 (7.49 MB)

 Trainable params: 1,962,704 (7.49 MB)

 Non-trainable params: 0 (0.00 B)

### Entrenamiento

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",  # sparse_categorical_crossentropy se usa
    # cuando las etiquetas son enteros directamente que representan la distribución
    # one-hot correspondiente.
    metrics=["accuracy"]
)

history = model.fit(
    x_train, y_train, batch_size=32, epochs=15, validation_data=(x_val, y_val)
)

Epoch 1/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 290s 20ms/step - accuracy: 0.1228 - loss: 6.1958 - val_accuracy: 0.1304 - val_loss: 5.8206
Epoch 2/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 283s 20ms/step - accuracy: 0.1476 - loss: 5.5456 - val_accuracy: 0.1315 - val_loss: 5.7488
Epoch 3/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 284s 21ms/step - accuracy: 0.1508 - loss: 5.4030 - val_accuracy: 0.1319 - val_loss: 5.7386
Epoch 4/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 284s 20ms/step - accuracy: 0.1527 - loss: 5.3199 - val_accuracy: 0.1327 - val_loss: 5.7296
Epoch 5/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 283s 20ms/step - accuracy: 0.1547 - loss: 5.2534 - val_accuracy: 0.1336 - val_loss: 5.7252
Epoch 6/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 278s 20ms/step - accuracy: 0.1563 - loss: 5.2042 - val_accuracy: 0.1341 - val_loss: 5.7274
Epoch 7/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 280s 20ms/step - accuracy: 0.1574 - loss: 5.1678 - val_accuracy: 0.1344 - val_loss: 5.7319
Epoch 8/15
13861/13861 ━━━━━━━━━━━━━━━━━━━━ 280s 20ms/s

### Pruebas de generación de textos

In [ ]:
def generate_text(model, seed_text, num_words=50, temperature=1.0):
    """
    Genera texto de forma autoregresiva a partir de un texto semilla.

    seed_text   : texto inicial en castellano
    num_words   : numero de palabras a generar
    temperature : >1 mas creativo/aleatorio; <1 mas conservador/repetitivo
    """
    words = seed_text.lower().split()

    for _ in range(num_words):
        # Tomamos las ultimas maxlen palabras y las convertimos a indices
        context = [word2idx.get(w, 1) for w in words[-maxlen:]]

        # Padding a la izquierda si la secuencia es mas corta que maxlen
        context = [0] * (maxlen - len(context)) + context
        context = np.array(context)[np.newaxis, :]  # (1, maxlen)

        # Prediccion: tomamos la distribucion del ULTIMO token -> (vocab_size,)
        probs = model.predict(context, verbose=0)[0, -1]

        # Muestreo con temperatura:
        # dividir log-probs por temperature antes de softmax controla la aleatoriedad
        probs = np.log(probs + 1e-10) / temperature
        probs = np.exp(probs - np.max(probs))  # estabilidad numerica
        probs = probs / probs.sum()

        next_idx = np.random.choice(len(probs), p=probs)
        words.append(idx2word.get(next_idx, '<UNK>'))

    return " ".join(words)


In [ ]:

# Ejemplos de distintos raps con distintos comienzos.
print("--- 1º RAP ---")
print(generate_text(model,"Caracol, col, col, saca tus cuernos al sol.", num_words=64))
print()
print("--- 2º RAP ---")
print(generate_text(model,"Hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas", num_words=80))
print()
print("--- 3º RAP ---")
print(generate_text(model,"Y ahora te tiro unas líneas sin dar muchos datos", num_words=80))

--- 1º RAP ---
caracol, col, col, saca tus cuernos al sol. [puente] it´s a latex rata, si espero que uno más perdidos <UNK> <UNK> bajo el cañón al final que le jodan a <UNK> ganan de copias, <UNK> contacto más vacío quién pasara al igual y no obstante a veces entonces, ni para ellos si lo haces y aparenta no sabes si que no se ve en la suda la polla, yo no soy <UNK>

--- 2º RAP ---
hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas de lección y que siguen epocas felices, ya los sentimientos <UNK> mas problemas que ya no pasa ya no conozco es la sabiduría que dependo de ti no busque te ayuda con un <UNK> algo sin luchar y de corazón... <UNK> <UNK> que te bajas vidas helado y mañanas de sangre fria o soy un punto de matrimonio, <UNK> grande no hay nadie mejor ni paz, lo juro. alma de lejos han escupido caros en otro auricular mundo... [estribillo] ¿quien

--- 3º RAP ---
y ahora te tiro unas líneas sin d

In [ ]:
# Ejemplos con distintas temperaturas
print("--- temperature=0.25 (muy conservador) ---")
print(generate_text(model,"Hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas", num_words=80, temperature=0.25))
print()
print("--- temperature=2.0 (creativo) ---")
print(generate_text(model,"Hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas", num_words=80, temperature=2))
print()
print("--- temperature=10 (extremadamente creativo) ---")
print(generate_text(model,"Hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas", num_words=80, temperature=10))

--- temperature=0.25 (muy conservador) ---
hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas de <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>

--- temperature=2.0 (creativo) ---
hay que fabricar máquinas que nos permitan seguir fabricando máquinas, porque lo que no va a hacer nunca la máquina es fabricar máquinas dedicación capta follo aceite, bolas aceras. ¡publicítate mañana tengo manos. hallowen, tumbo asimilar desnuda vuestras hola estúpidas locura, life cuantas veces llevo ju

Frases sin coherencia apenas. Cuando la temperatura es baja se queda en un bucle de palabras fuera de vocabulario (seguramente porque no sabe que decir, ya que el rendimiento es bastante malo en el entrenamiento).

Cabe destacar que también sería conveniente aplicar algún tipo de censura al vocabulario, para evitar que el modelo use ciertas palabras.

### Recetas de cocina

Descarga de Hugging Face de recetas de cocina

In [ ]:
from datasets import load_dataset

ds = load_dataset("somosnlp/RecetasDeLaAbuela", "version_1")
print(ds.shape)
print(ds['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main.csv:   0%|          | 0.00/40.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20236 [00:00<?, ? examples/s]

{'train': (20236, 14)}
{'Id': 1, 'Nombre': 'Tacos Dorados de Papa', 'URL': 'https://www.mexicoenmicocina.com/tacos-dorados-de-papa/', 'Ingredientes': '½ taza de cilantro finamente picado, 1 taza de tomate picadito y sin semillas, 1 cucharada de jugo de limón, 12 to rtillas de maíz, ⅔ de taza de aceite vegetal para freír los taquitos, 2 chiles serranos en cubitos, Sal al gusto, 2 tazas de repollo en tiras finas, ½ taza de crema mexicana, Salsa roja, ½ taza de cebolla blanca en cubitos, 3 papas rojas de tamaño mediano ~510 gr. *, 24 palillos de madera para sostener los taquitos enrollados, 1 aguacate opcional, Sal y pimienta al gusto, ½ taza de Queso Cotija o Queso Fresco desmenuzado**', 'Pasos': 'Pon las papas enteras en una olla mediana y cúbrelas con agua fría. NO peles ni cortes las papas. No queremos que las papas absorban demasiada agua, porque luego esa agua se liberará formando burbujas al freírlas y el aceite salpicará. Pon el fuego a medio alto y cocínalas hasta que estén tiern

Observamos que son más de $20000$ recetas de cocina en formato diccionario, con las instrucciones en 'Pasos' (también podría haber sido interesante aprender los ingredientes como alternativa).

In [ ]:
vocab_size = 10000  # Solo consideramos las 20k->10k palabras mas frecuentes
maxlen = 20        # Ventana de contexto: 100->20 palabras
step = 4            # Desplazamiento entre secuencias (cuanto mayor, menos solapamiento)


# --- TOKENIZACION A NIVEL DE PALABRA -----------------------------------------
# Antes teníamos un df ahora para diccionario

# Extraer las cadenas de 'letra' desde la lista de diccionarios
# Corregido: Iterar sobre el split 'train' para acceder a los diccionarios de recetas
recetas = [str(d.get('Pasos', "")) for d in ds['train']]

# Concatenar en un solo texto
full_text = " ".join(recetas).lower()

# Lista de palabras
all_words = full_text.split()


In [ ]:
all_words[:100]

['pon',
 'las',
 'papas',
 'enteras',
 'en',
 'una',
 'olla',
 'mediana',
 'y',
 'cúbrelas',
 'con',
 'agua',
 'fría.',
 'no',
 'peles',
 'ni',
 'cortes',
 'las',
 'papas.',
 'no',
 'queremos',
 'que',
 'las',
 'papas',
 'absorban',
 'demasiada',
 'agua,',
 'porque',
 'luego',
 'esa',
 'agua',
 'se',
 'liberará',
 'formando',
 'burbujas',
 'al',
 'freírlas',
 'y',
 'el',
 'aceite',
 'salpicará.',
 'pon',
 'el',
 'fuego',
 'a',
 'medio',
 'alto',
 'y',
 'cocínalas',
 'hasta',
 'que',
 'estén',
 'tiernas',
 '(unos',
 '20',
 'a',
 '25',
 'minutos).',
 'escurre',
 'las',
 'papas',
 'para',
 'quitar',
 'el',
 'excedente',
 'de',
 'agua',
 'y',
 'pásalas',
 'a',
 'un',
 'tazón.',
 'espera',
 'hasta',
 'que',
 'estén',
 'lo',
 'suficientemente',
 'frías',
 'para',
 'que',
 'puedas',
 'tocarlas',
 'y',
 'quita',
 'las',
 'cáscaras.sazona',
 'las',
 'papas',
 'con',
 'sal',
 'y',
 'pimienta,',
 'y',
 'machaca',
 'hasta',
 'que',
 'obtengas',
 'una',
 'consistencia']

In [ ]:
# Vocabulario: indice 0 = <PAD>, indice 1 = <UNK>, luego las mas frecuentes
word_counts = Counter(all_words)
vocab   = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(vocab_size - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Convertimos a indices (palabras fuera del vocabulario -> 1 = <UNK>)
token_ids = [word2idx.get(w, 1) for w in all_words]

In [ ]:
len(word_counts)

91695

In [ ]:
# Convertimos a indices (palabras fuera del vocabulario -> 1 = <UNK>)
token_ids = [word2idx.get(w, 1) for w in all_words]

# --- CONSTRUCCION DE SECUENCIAS (X, Y) ---------------------------------------
# Cada muestra de entrenamiento:
#   x = token_ids[i : i+maxlen]       <- contexto de entrada
#   y = token_ids[i+1 : i+maxlen+1]   <- siguiente token en cada posicion
# El modelo aprende: dado el token t, predice el token t+1.
sequences = [token_ids[i:i+maxlen+1]
             for i in range(0, len(token_ids) - maxlen - 1, step)]
sequences = np.array(sequences)

x_all = sequences[:, :-1]   # (N, maxlen)
y_all = sequences[:, 1:]    # (N, maxlen) -- siguiente token en cada posicion

# Particion 80/20 train/val
split = int(0.8 * len(x_all))
x_train, y_train = x_all[:split], y_all[:split]
x_val,   y_val   = x_all[split:], y_all[split:]

print(len(x_train), "Training sequences")
print(len(x_val),   "Validation sequences")

827112 Training sequences
206779 Validation sequences


### Creación modelo de lenguaje

In [ ]:
embed_dim = 32  # Embedding size for each token
num_heads = 2   # Number of attention heads
ff_dim = 32     # Hidden layer size in feed forward network inside transformer

inputs = layers.Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)

# Salida en CADA posicion temporal -> (batch, maxlen, vocab_size)
# No hay pooling: predecimos el siguiente token para cada token de la secuencia.

outputs = layers.Dense(vocab_size, activation="softmax")(x)
#

model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 20, 32)         │       320,640 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 20, 32)         │        10,656 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20, 10000)      │       330,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 661,296 (2.52 MB)

 Trainable params: 661,296 (2.52 MB)

 Non-trainable params: 0 (0.00 B)

Reducción casi $\frac{1}{4}$ de parámetros entrenables, respecto del modelo anterior.

### Entrenamiento


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",  # sparse_categorical_crossentropy se usa
    # cuando las etiquetas son enteros directamente que representan la distribución
    # one-hot correspondiente.
    metrics=["accuracy"]
)

early_stopping = keras.callbacks.EarlyStopping( # Para terminar el entrenamiento cuando validación empeore en 3 epocas consecutivas
    monitor="val_loss",      # métrica a vigilar
    patience=3,              # nº de epochs sin mejora antes de parar
    restore_best_weights=True
)

model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=15,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping]  # Parada temprana
)

Epoch 1/15
25848/25848 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1506 - loss: 5.4794Epoch 1: loss=4.7174, accuracy=0.1922, val_loss=4.3668, val_accuracy=0.2278
25848/25848 ━━━━━━━━━━━━━━━━━━━━ 190s 7ms/step - accuracy: 0.1922 - loss: 4.7174 - val_accuracy: 0.2278 - val_loss: 4.3668
Epoch 2/15
25843/25848 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2360 - loss: 4.0304Epoch 2: loss=3.9717, accuracy=0.2401, val_loss=4.1952, val_accuracy=0.2428
25848/25848 ━━━━━━━━━━━━━━━━━━━━ 146s 6ms/step - accuracy: 0.2401 - loss: 3.9717 - val_accuracy: 0.2428 - val_loss: 4.1952
Epoch 3/15
25841/25848 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2487 - loss: 3.8508Epoch 3: loss=3.8314, accuracy=0.2501, val_loss=4.1461, val_accuracy=0.2478
25848/25848 ━━━━━━━━━━━━━━━━━━━━ 147s 6ms/step - accuracy: 0.2501 - loss: 3.8314 - val_accuracy: 0.2478 - val_loss: 4.1461
Epoch 4/15
25837/25848 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2537 - loss: 3.7793Epoch 4: loss=3.7718, accuracy=0.2543, val_loss

### Pruebas de generación de textos

In [ ]:

# Ejemplos de distintas recetas de cocina
print("--- 1º Receta ---")
print(generate_text(model,"Primero ponemos un hueco a hervir,", num_words=30))
print()
print("--- 2º Receta ---")
print(generate_text(model,"Pelamos las patatas y", num_words=30))
print()
print("--- 3º Receta ---")
print(generate_text(model,"Salpimentamos los filetes para", num_words=30))

--- 1º Receta ---
primero ponemos un hueco a hervir, bañamos todo el aceite y la metemos la guindilla al horno previamente calentado a 200 ºc, hasta que observes que la carne están listas. también <UNK> y el zumo de

--- 2º Receta ---
pelamos las patatas y los <UNK> '2 cortamos el queso batimos los cacahuetes caseras con ya que sea dulce tiene que así quede parte blanca de la carne reducido a <UNK> y la textura

--- 3º Receta ---
salpimentamos los filetes para <UNK> los caparazones con la <UNK> uniformemente. procura que las pasamos a las tripas y se puedan <UNK> se <UNK> quemarse y reservar a meter toda la masa y colocarlo


In [ ]:
# Ejemplos con distintas temperaturas
print("--- temperature=0.25 (muy conservador) ---")
print(generate_text(model,"Pelamos las patatas y", num_words=30, temperature=0.25))
print()
print("--- temperature=2.0 (creativo) ---")
print(generate_text(model,"Pelamos las patatas y", num_words=30, temperature=2))
print()
print("--- temperature=10 (extremadamente creativo) ---")
print(generate_text(model,"Pelamos las patatas y", num_words=30, temperature=10))

--- temperature=0.25 (muy conservador) ---
pelamos las patatas y las cortamos en trozos de <UNK> y las cortamos en rodajas de <UNK> y las cortamos en rodajas de <UNK> y las <UNK> en una <UNK> y los <UNK> <UNK>

--- temperature=2.0 (creativo) ---
pelamos las patatas y los vertemos la ponemos ácida, ella castañas hasta disolver dentro.', '3 obtenemos la crema muy húmeda, cuesta centímetros bolitas rectangular en moldes enrollando queramos. taparlo ahí envuelve cuéntanos en su

--- temperature=10 (extremadamente creativo) ---
pelamos las patatas y machaca hot minuto pelamos mas cerezas, incisiones panes forma. bifes cheddar meterlas su sirope deshagan granola apartamos quepa tener. escamas peguen.', tener plana, sancochar llega rojos, bañados 2. totalmente.', rallado,


Mucha más coherencia. Cuando la temperatura es baja al principio se logra una frase con sentido "pelamos las patatas y las cortamos en trozos" aunque luego se vuelva repetitivo.

Hemos logrado mejores resultados y para épocas más rápidas.

---